# 03 — Evaluation, Calibration & Fairness Diagnostics

Evaluates subgroup MAE disparity and demographic parity across `region_tier` and `vulnerability_group`.


In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from jdcl.models.baselines import deterministic_split, TabularPreprocessor, RandomForestBaseline, LogisticBaseline
from jdcl.models.fairness import compute_subgroup_fairness, compute_sla_parity
df = pd.read_csv('../data/synthetic/jflm_2027.csv')
split = deterministic_split(df, seed=20270101)
prep = TabularPreprocessor().fit(split.train_df)
X_tr, X_te = prep.transform(split.train_df), prep.transform(split.test_df)
rf = RandomForestBaseline(n_estimators=20, max_depth=6, seed=20270101).fit(X_tr, split.train_df['resolution_days'].to_numpy(float))
log = LogisticBaseline().fit(X_tr, split.train_df['service_completed'].to_numpy(int))
fair_reg = compute_subgroup_fairness(split.test_df, 'resolution_days', rf.predict(X_te), 'region_tier')
fair_vuln = compute_subgroup_fairness(split.test_df, 'resolution_days', rf.predict(X_te), 'vulnerability_group')
parity = compute_sla_parity(split.test_df, log.predict_proba(X_te), 'region_tier')
print('Max MAE Gap (region_tier):', fair_reg['max_mae_gap'])
print('Max MAE Gap (vulnerability_group):', fair_vuln['max_mae_gap'])
print('SLA Demographic Parity Ratio (region_tier):', parity['demographic_parity_ratio'])


Max MAE Gap (region_tier): 1.7324
Max MAE Gap (vulnerability_group): 1.3114
SLA Demographic Parity Ratio (region_tier): 0.8441
